# A2 Pileup Multi-Task Tuning (CNN + Dense Classifier)

Grid search over CNN autoencoder + dense classifier head, using the **same
hyperparameter axes as the Singles tuning notebook**:

- `latent_dim` (fixed at **16** for pileup — more capacity than singles)
- `conv_filters`: `[(16,32,64), (32,64,128), (64,128,256)]`
- `dense_units`: `[16, 32, 64, 128]`  (classifier head width)
- `lr`: `[1e-2, 1e-3, 1e-4]`
- `num_layers`: `[2, 3, 4]` (encoder/decoder depth)

**Total: 1 × 3 × 4 × 3 × 3 = 108 combinations.**

Multi-task model: CNN autoencoder branch (reconstruction, MSE) + dense classifier
head (2 sigmoid outputs for primary/secondary labels, BCE). Trained jointly with
`loss_weights={reconstruction: 0.9, classification: 0.1}` to match the Singles setup.

In [1]:
import itertools
import random
import time

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, regularizers

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix

import matplotlib.pyplot as plt

tf.config.optimizer.set_jit(False)


print(f"TensorFlow {tf.__version__}")
print(f"GPUs: {tf.config.list_physical_devices('GPU')}")

import gc

I0000 00:00:1776306135.734092 3468724 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776306135.756166 3468724 pjrt_api.cc:96] PJRT_Api is set for device type gpu


I0000 00:00:1776306136.297894 3468724 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow 2.22.0-dev0+selfbuilt
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Load and split data

Same pipeline as the baseline autoencoder notebook: load Euclidean-normalized
waveforms, split 60/20/20 train/val/test, then min-max normalize to [0, 1]
(matching the sigmoid output activation).

In [2]:
# Load pileup waveforms. Labels are (primary, secondary) binary (0=photon, 1=neutron).
d = np.load("pileup_waveforms.npz")
X = d["pileup_wf"].astype(np.float32)
y_primary   = d["primary_label"].astype(np.int32)
y_secondary = d["secondary_label"].astype(np.int32)
Y = np.column_stack([y_primary, y_secondary]).astype(np.float32)  # (N, 2)

# 60/20/20 train/val/test split
X_tv, X_test, Y_tv, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42,
)
X_train, X_val, Y_train, Y_val = train_test_split(
    X_tv, Y_tv, test_size=0.25, random_state=42,
)

# MinMax scale to [0, 1] using only training statistics
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val   = scaler.transform(X_val).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)

input_length = X_train.shape[1]

X_train = X_train[..., None]
X_val   = X_val[..., None]
X_test  = X_test[..., None]


print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
print(f"Label shape: {Y_train.shape}")
print(f"Input length: {input_length}")
print(f"Value range: [{X_train.min():.4f}, {X_train.max():.4f}]")


Train: (104001, 104, 1)  Val: (34668, 104, 1)  Test: (34668, 104, 1)
Label shape: (104001, 2)
Input length: 104
Value range: [0.0000, 1.0000]


## Build multi-task CNN model

Same architecture pattern as Singles: CNN encoder → latent Dense → CNN decoder
(mirrored) + dense classifier head. Classifier outputs **2 sigmoid units**
(primary + secondary labels). Lambda-crops decoder output to input length to
handle odd pooling/upsampling asymmetry at `num_layers=4`.

In [3]:
def resolve_filters(conv_filters, num_layers):
    """Option A: truncate if fewer layers, extend by doubling if more."""
    f = list(conv_filters)
    if num_layers <= len(f):
        return tuple(f[:num_layers])
    while len(f) < num_layers:
        f.append(f[-1] * 2)
    return tuple(f)


def build_model(latent_dim=16,
                conv_filters=(32, 64, 128),
                dense_units=32,
                lr=1e-3,
                num_layers=3):

    filters = resolve_filters(conv_filters, num_layers)

    inputs = tf.keras.Input(shape=(input_length, 1))

    # Encoder
    x = inputs
    for f in filters:
        x = layers.Conv1D(f, 3, activation='relu', padding='same')(x)
        x = layers.MaxPooling1D(2, padding='same')(x)

    shape = x.shape[1:]

    x = layers.Flatten()(x)
    latent = layers.Dense(latent_dim, name="latent")(x)

    # Decoder (mirror)
    x = layers.Dense(shape[0] * shape[1], activation='relu')(latent)
    x = layers.Reshape(shape)(x)
    for f in reversed(filters):
        x = layers.UpSampling1D(2)(x)
        x = layers.Conv1D(f, 3, activation='relu', padding='same')(x)

    # Crop decoder output to input_length using Cropping1D (avoids Lambda + Keras 3 graph issues)
    decoder_length = int(shape[0] * (2 ** num_layers))
    crop_amount = decoder_length - input_length
    if crop_amount > 0:
        x = layers.Cropping1D(cropping=(0, crop_amount))(x)

    recon_conv = layers.Conv1D(1, 3, padding='same')(x)
    reconstruction = layers.Reshape((input_length,), name="reconstruction")(recon_conv)

    # Classifier head (matches OLD a2_multitask: two hidden dense layers, no dropout)
    clf = layers.Dense(dense_units, activation='relu')(latent)
    clf = layers.Dense(max(2, dense_units // 2), activation='relu')(clf)
    classification = layers.Dense(2, activation='sigmoid', name="classification")(clf)

    model = Model(inputs, {"reconstruction": reconstruction, "classification": classification})

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss={
            "reconstruction": "mse",
            "classification": "binary_crossentropy",
        },
        loss_weights={"reconstruction": 1000.0, "classification": 1.0},
        metrics={"classification": [tf.keras.metrics.BinaryAccuracy(name="binary_accuracy")]},
        jit_compile=False
    )

    return model


## Hyperparameter grid (108 combinations)

In [4]:
param_grid = {
    "latent_dim":   [16, 32],  # swept (32 matches a2_multitask tuned)
    "conv_filters": [(16, 32, 64), (32, 64, 128), (64, 128, 256)],
    "dense_units":  [16, 32, 64, 128],
    "lr":           [1e-2, 1e-3, 1e-4],
    "num_layers":   [2, 3, 4],
}

def param_combinations():
    keys = list(param_grid.keys())
    for values in itertools.product(*param_grid.values()):
        yield dict(zip(keys, values))

all_params = list(param_combinations())
print(f"Total combinations: {len(all_params)}")

Total combinations: 216


## Callbacks and results setup

In [5]:
import csv, os

best_score = -np.inf
best_model = None
best_params = None

results_path = "DoublesApril14_hyperparam_results.csv"
fieldnames = ["trial", "latent_dim", "conv_filters", "resolved_filters",
              "dense_units", "lr", "num_layers",
              "best_val_binary_acc", "best_val_recon_loss", "best_val_loss", "epochs_run"]

completed_keys = set()
if os.path.exists(results_path):
    # Load existing rows (any old schema)
    with open(results_path, newline="") as f:
        old_rows = list(csv.DictReader(f))

    # Normalize each row to new schema: ensure both conv_filters (original) and resolved_filters
    for row in old_rows:
        cf_str = row.get("conv_filters", "")
        cf = eval(cf_str) if cf_str else None
        nl = int(row["num_layers"])

        # If "resolved_filters" missing, compute it from conv_filters + num_layers
        if "resolved_filters" not in row or not row.get("resolved_filters"):
            # Determine original tuple: if conv_filters has 3 entries it's the grid original
            if cf is not None and len(cf) == 3:
                original = cf
            else:
                # Already-resolved tuple — try to match back to a grid original
                original = None
                for orig in [(16, 32, 64), (32, 64, 128), (64, 128, 256)]:
                    if resolve_filters(orig, nl) == tuple(cf):
                        original = orig
                        break
                if original is None:
                    original = tuple(cf)  # fallback
            row["conv_filters"] = str(original)
            row["resolved_filters"] = str(resolve_filters(original, nl))

    # Rewrite the CSV with the normalized schema
    with open(results_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in old_rows:
            # Only write keys we care about; ignore stale columns
            w.writerow({k: r.get(k, "") for k in fieldnames})

    # Build completed_keys (using ORIGINAL conv_filters since loop iterates originals)
    for row in old_rows:
        key = (int(row["latent_dim"]), row["conv_filters"], int(row["dense_units"]),
               float(row["lr"]), int(row["num_layers"]))
        completed_keys.add(key)
        acc = float(row["best_val_binary_acc"])
        if acc > best_score:
            best_score = acc
            best_params = {
                "latent_dim":   int(row["latent_dim"]),
                "conv_filters": eval(row["conv_filters"]),
                "dense_units":  int(row["dense_units"]),
                "lr":           float(row["lr"]),
                "num_layers":   int(row["num_layers"]),
            }
    print(f"Resuming: {len(completed_keys)} trials already done. Best so far: {best_score}")
else:
    with open(results_path, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=fieldnames).writeheader()
    print("Starting fresh")


Resuming: 145 trials already done. Best so far: 0.9428435564041138


## Tuning loop

In [ ]:
for i, params in enumerate(all_params):
    key = (params["latent_dim"], str(params["conv_filters"]), params["dense_units"],
           params["lr"], params["num_layers"])
    if key in completed_keys:
        print(f"\nTrial {i+1}/{len(all_params)}: SKIP (already done) {params}")
        continue

    tf.keras.backend.clear_session()
    gc.collect()
    print(f"\nTrial {i+1}/{len(all_params)}")

    # Resolve actual conv filters used (truncate/extend based on num_layers)
    resolved = resolve_filters(params["conv_filters"], params["num_layers"])

    try:
        model = build_model(**params)

        # Fresh callbacks per trial (avoids state leak across trials)
        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True, mode="min"
        )
        checkpoint = tf.keras.callbacks.ModelCheckpoint(
            f"trial_checkpoints/best_model_a2_trial{i+1}.keras",
            monitor="val_classification_binary_accuracy",
            save_best_only=True, mode="max"
        )

        history = model.fit(
            X_train,
            {"reconstruction": X_train, "classification": Y_train},
            validation_data=(X_val, {"reconstruction": X_val, "classification": Y_val}),
            epochs=200,
            batch_size=256,
            callbacks=[early_stop, checkpoint],
            verbose=1
        )

        h = history.history
        val_acc = max(h["val_classification_binary_accuracy"])

        row = {
            "trial": i + 1,
            "latent_dim": params["latent_dim"],
            "conv_filters": str(params["conv_filters"]),  # original grid tuple (for resume key)
            "resolved_filters": str(resolved),             # what the model actually used
            "dense_units": params["dense_units"],
            "lr": params["lr"],
            "num_layers": params["num_layers"],
            "best_val_binary_acc": val_acc,
            "best_val_recon_loss": min(h["val_reconstruction_loss"]),
            "best_val_loss": min(h["val_loss"]),
            "epochs_run": len(h["val_loss"]),
        }
        with open(results_path, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=fieldnames).writerow(row)

        print("Params:", params, "resolved filters:", resolved)
        print("Best val binary acc:", val_acc)

        if val_acc > best_score:
            best_score = val_acc
            best_model = model
            best_params = params

    except Exception as e:
        print(f"Trial {i+1} CRASHED: {type(e).__name__}: {e}")
        print("Continuing to next trial — this one will be retried on resume.")
        continue


## Best result

In [7]:
print("BEST SCORE (val binary acc):", best_score)
print("BEST PARAMS:", best_params)

BEST SCORE (val binary acc): 0.9438819885253906
BEST PARAMS: {'latent_dim': 32, 'conv_filters': (64, 128, 256), 'dense_units': 128, 'lr': 0.0001, 'num_layers': 4}
